# 05 分層分析與干擾因子 — 練習

用松柏護理之家退伍軍人症 line list 練習分層分析和 Mantel-Haenszel 法。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

## 題目 1：水療使用的干擾分析

Ch03 發現 `hydrotherapy_use` 也與感染有關。現在懷疑 `functional_status` 同樣是它的干擾因子。

1. 計算 `hydrotherapy_use → infected` 的粗 RR
2. 驗證干擾三要件：`functional_status` 與 `hydrotherapy_use` 有關嗎？（用 crosstab）
3. 按 `functional_status` 分層，計算各層的 RR 和 95% CI
4. 比較：各層 RR 和粗 RR 的差異

In [ ]:
# TODO: 粗 RR (hydrotherapy_use → infected)
# TODO: 驗證干擾條件
# TODO: 分層 RR

## 題目 2：Mantel-Haenszel 調整

承題目 1，對 `hydrotherapy_use → infected`（控制 `functional_status`）計算 MH 調整後 RR。

1. 用公式 $RR_{MH} = \frac{\sum_i a_i(c_i+d_i)/N_i}{\sum_i c_i(a_i+b_i)/N_i}$ 手算
2. 比較粗 RR 和 MH RR，差異大嗎？
3. 結論：功能狀態是否為水療使用的干擾因子？

In [ ]:
# TODO: 計算 MH 調整後 RR
# TODO: 比較粗 RR vs MH RR

## 題目 3（挑戰題）：按年齡組分層 + 森林圖

1. 建立 `age_group`（60-69 / 70-79 / 80-89 / 90+）
2. 按 `age_group` 分層，計算 `shower_use → infected` 的各層 RR 和 95% CI
3. 畫森林圖（forest plot），紅色虛線標出粗 RR
4. 計算 MH 調整後 RR
5. 年齡是淋浴使用的干擾因子嗎？是否有交互作用？

In [ ]:
# TODO: 建立 age_group
# TODO: 分層 RR + 95% CI
# TODO: 森林圖
# TODO: MH 調整後 RR
# TODO: 解讀

## 題目 4：共病與年齡對死亡率的干擾（COVID-19 情境）

某醫院收治的 COVID-19 病例，line list 記錄了年齡層（`age_group`）、是否有共病（`comorbidity`，如糖尿病、心血管疾病等）與死亡結果（`death`）。你懷疑年齡是共病影響死亡率的干擾因子。

1. 計算 `comorbidity → death` 的粗 RR
2. 驗證干擾三要件：`age_group` 是否分別與 `comorbidity`、`death` 有關？（用 crosstab）
3. 按 `age_group` 分層，計算各層的 RR 及 95% CI
4. 計算 Mantel-Haenszel 調整後 RR，並與粗 RR 比較
5. 解讀：年齡是否為共病影響死亡率的干擾因子？調整後的結論和粗分析有何不同？

In [ ]:
# --- 資料：COVID-19 病例的年齡與共病 ---
rng = np.random.default_rng(20)
n = 800

age_group = rng.choice(["under60", "60plus"], size=n, p=[0.6, 0.4])
comorbidity = np.array([
    rng.binomial(1, 0.5 if ag == "60plus" else 0.15) for ag in age_group
])
death_prob = np.select(
    [
        (age_group == "under60") & (comorbidity == 0),
        (age_group == "under60") & (comorbidity == 1),
        (age_group == "60plus") & (comorbidity == 0),
        (age_group == "60plus") & (comorbidity == 1),
    ],
    [0.02, 0.05, 0.12, 0.30],
)
death = rng.binomial(1, death_prob)

covid_df = pd.DataFrame({
    "case_id": [f"C{i:04d}" for i in range(n)],
    "age_group": age_group,
    "comorbidity": comorbidity,
    "death": death,
})

# TODO: 計算 comorbidity -> death 的粗 RR
# TODO: 驗證干擾三要件：age_group 是否分別與 comorbidity、death 有關？
# TODO: 按 age_group 分層，計算各層的 RR 及 95% CI
# TODO: 計算 Mantel-Haenszel 調整後 RR，並與粗 RR 比較
# TODO: 解讀：年齡是否為共病影響死亡率的干擾因子？

## 題目 5：疫苗接種的干擾分析（流感情境）

社區流感監測資料記錄了年齡層（`age_group`）、是否接種流感疫苗（`vaccinated`）與是否感染流感（`infected`）。長者屬高風險族群，接種率較高，但長者本身感染風險也較高——這是典型的「適應症干擾（confounding by indication）」。

1. 計算 `vaccinated → infected` 的粗 RR
2. 驗證干擾三要件：`age_group` 是否分別與 `vaccinated`、`infected` 有關？
3. 按 `age_group` 分層，計算各層的 RR 及 95% CI
4. 計算 Mantel-Haenszel 調整後 RR
5. 解讀：粗 RR 和 MH RR 何者更接近疫苗的「真實」保護效果？為什麼粗分析容易低估疫苗效果？

In [ ]:
# --- 資料：流感疫苗接種與年齡 ---
rng = np.random.default_rng(11)
n = 800

age_group = rng.choice(["under65", "65plus"], size=n, p=[0.7, 0.3])
vaccinated = np.array([
    rng.binomial(1, 0.7 if ag == "65plus" else 0.3) for ag in age_group
])
infect_prob = np.select(
    [
        (age_group == "under65") & (vaccinated == 0),
        (age_group == "under65") & (vaccinated == 1),
        (age_group == "65plus") & (vaccinated == 0),
        (age_group == "65plus") & (vaccinated == 1),
    ],
    [0.20, 0.10, 0.40, 0.20],
)
infected = rng.binomial(1, infect_prob)

flu_df = pd.DataFrame({
    "case_id": [f"F{i:04d}" for i in range(n)],
    "age_group": age_group,
    "vaccinated": vaccinated,
    "infected": infected,
})

# TODO: 計算 vaccinated -> infected 的粗 RR
# TODO: 驗證干擾三要件：age_group 是否分別與 vaccinated、infected 有關？
# TODO: 按 age_group 分層，計算各層的 RR 及 95% CI
# TODO: 計算 Mantel-Haenszel 調整後 RR
# TODO: 解讀：粗 RR 是否低估了疫苗的保護效果？

## 題目 6：食物暴露的干擾分析（甲型肝炎情境）

某聚餐活動後爆發甲型肝炎群聚，line list 記錄了是否曾接種 A 肝疫苗（`vaccinated`）、是否食用生蠔（`ate_shellfish`）與是否感染（`infected`）。你懷疑疫苗接種史干擾了食物暴露與感染的關聯。

1. 計算 `ate_shellfish → infected` 的粗 RR
2. 驗證干擾三要件：`vaccinated` 是否分別與 `ate_shellfish`、`infected` 有關？
3. 按 `vaccinated` 分層，計算各層的 RR 及 95% CI
4. 計算 Mantel-Haenszel 調整後 RR，並與粗 RR 比較
5. 解讀：疫苗接種史是否為生蠔食用與感染關聯的干擾因子？

In [ ]:
# --- 資料：甲型肝炎聚餐群聚事件 ---
rng = np.random.default_rng(5)
n = 900

vaccinated = rng.binomial(1, 0.35, size=n)
ate_shellfish = np.array([
    rng.binomial(1, 0.3 if v == 1 else 0.6) for v in vaccinated
])
infect_prob = np.select(
    [
        (vaccinated == 0) & (ate_shellfish == 0),
        (vaccinated == 0) & (ate_shellfish == 1),
        (vaccinated == 1) & (ate_shellfish == 0),
        (vaccinated == 1) & (ate_shellfish == 1),
    ],
    [0.05, 0.35, 0.01, 0.07],
)
infected = rng.binomial(1, infect_prob)

hav_df = pd.DataFrame({
    "case_id": [f"H{i:04d}" for i in range(n)],
    "vaccinated": vaccinated,
    "ate_shellfish": ate_shellfish,
    "infected": infected,
})

# TODO: 計算 ate_shellfish -> infected 的粗 RR
# TODO: 驗證干擾三要件：vaccinated 是否分別與 ate_shellfish、infected 有關？
# TODO: 按 vaccinated 分層，計算各層的 RR 及 95% CI
# TODO: 計算 Mantel-Haenszel 調整後 RR，並與粗 RR 比較
# TODO: 解讀：疫苗接種史是否為食物暴露的干擾因子？

## 題目 7：依區域分層分析（登革熱情境）

某縣市登革熱疫情調查，line list 記錄了居住區域（`region`：urban／suburban／rural）、住家是否有積水容器（`standing_water`）與是否感染登革熱（`infected`）。不同區域的病媒蚊密度與居民積水管理習慣不同，你懷疑地區是干擾因子。

1. 計算 `standing_water → infected` 的粗 RR
2. 驗證干擾三要件：`region` 是否分別與 `standing_water`、`infected` 有關？
3. 按 `region` 分層，計算各層的 RR 及 95% CI
4. 計算 Mantel-Haenszel 調整後 RR
5. 解讀：地區是否為積水暴露與感染登革熱關聯的干擾因子？

In [ ]:
# --- 資料：登革熱跨區域調查 ---
rng = np.random.default_rng(42)
n = 900

region = rng.choice(["urban", "suburban", "rural"], size=n, p=[0.4, 0.35, 0.25])
water_p = {"urban": 0.2, "suburban": 0.4, "rural": 0.6}
standing_water = np.array([rng.binomial(1, water_p[r]) for r in region])
infect_p = {
    ("urban", 0): 0.03, ("urban", 1): 0.09,
    ("suburban", 0): 0.08, ("suburban", 1): 0.24,
    ("rural", 0): 0.15, ("rural", 1): 0.45,
}
infect_prob = np.array([infect_p[(r, w)] for r, w in zip(region, standing_water)])
infected = rng.binomial(1, infect_prob)

dengue_df = pd.DataFrame({
    "case_id": [f"D{i:04d}" for i in range(n)],
    "region": region,
    "standing_water": standing_water,
    "infected": infected,
})

# TODO: 計算 standing_water -> infected 的粗 RR
# TODO: 驗證干擾三要件：region 是否分別與 standing_water、infected 有關？
# TODO: 按 region 分層（urban / suburban / rural），計算各層的 RR 及 95% CI
# TODO: 計算 Mantel-Haenszel 調整後 RR
# TODO: 解讀：地區是否為積水暴露與感染登革熱關聯的干擾因子？

## 題目 8（挑戰題）：共病干擾分析（結核情境）

某結核病接觸者篩檢資料，line list 記錄了是否有糖尿病（`diabetes`）、是否與結核病例密切接觸（`close_contact`）與是否確診活動性結核（`active_tb`）。糖尿病會削弱免疫力、增加感染結核的風險，同時糖尿病患者的照護與居住型態也可能影響其接觸史，你懷疑糖尿病是干擾因子。

1. 計算 `close_contact → active_tb` 的粗 OR
2. 驗證干擾三要件：`diabetes` 是否分別與 `close_contact`、`active_tb` 有關？
3. 按 `diabetes` 分層，計算各層的 OR 及 95% CI
4. 畫森林圖（forest plot），紅色虛線標出粗 OR
5. 計算 Mantel-Haenszel 調整後 OR
6. 解讀：糖尿病是密切接觸與活動性結核關聯的干擾因子，還是效果修飾因子（effect modifier）？各層 OR 是否相近？

In [ ]:
# --- 資料：結核病接觸者篩檢 ---
from epi_learning.metrics import odds_ratio

rng = np.random.default_rng(291)
n = 900

diabetes = rng.binomial(1, 0.25, size=n)
close_contact = np.array([
    rng.binomial(1, 0.5 if d == 1 else 0.25) for d in diabetes
])
tb_prob = np.select(
    [
        (diabetes == 0) & (close_contact == 0),
        (diabetes == 0) & (close_contact == 1),
        (diabetes == 1) & (close_contact == 0),
        (diabetes == 1) & (close_contact == 1),
    ],
    [0.02, 0.10, 0.08, 0.32],
)
active_tb = rng.binomial(1, tb_prob)

tb_df = pd.DataFrame({
    "case_id": [f"T{i:04d}" for i in range(n)],
    "diabetes": diabetes,
    "close_contact": close_contact,
    "active_tb": active_tb,
})

# TODO: 計算 close_contact -> active_tb 的粗 OR
# TODO: 驗證干擾三要件：diabetes 是否分別與 close_contact、active_tb 有關？
# TODO: 按 diabetes 分層，計算各層的 OR 及 95% CI
# TODO: 畫森林圖（forest plot），紅色虛線標出粗 OR
# TODO: 計算 Mantel-Haenszel 調整後 OR
# TODO: 解讀：糖尿病是干擾因子還是效果修飾因子？各層 OR 是否相近？